# OpenTelemetry Operator

Telemetrie ist der Prozess der automatischen Erfassung und Übertragung von Daten von entfernten oder verteilten Systemen, um deren Leistung und Status zu überwachen, zu messen und zu verfolgen. Telemetriedaten liefern Echtzeit-Einblicke in die Funktionsweise verschiedener Anwendungskomponenten. Sie stellen die Daten für Observability-Tools bereit, die Entwicklern und Systemadministratoren helfen, das System zu beobachten, Fehler zu beheben und es zu optimieren, ohne jede Komponente manuell überprüfen zu müssen.

**OpenTelemetry (OTel) ist ein Open-Source-Projekt**, das standardisierte Tools und APIs zum Generieren, Sammeln und Exportieren von Telemetriedaten wie Traces, Metriken und Logs bereitstellt. Ziel ist es, Entwicklern umfassende Einblicke in Anwendungen zu ermöglichen und sie bei der Überwachung, Fehlerbehebung und Optimierung von Softwaresystemen zu unterstützen.

Die Hauptziele von OpenTelemtry sind:

* Einheitliche Telemetrie : Kombiniert Tracing, Logging und Metriken in einem einzigen Framework, das die Korrelation aller Daten ermöglicht und einen offenen Standard für Telemetriedaten etabliert.
* Anbieterneutralität : Integration mit verschiedenen Backends zur Datenverarbeitung.
* Plattformübergreifend : Unterstützt verschiedene Sprachen (Java, Python, Go usw.) und Plattformen und ist daher vielseitig für unterschiedliche Entwicklungsumgebungen einsetzbar.


---

### Installation

zuerst muss der Cert-Manager installiert werden

In [ ]:
%%bash
curl -sfL https://raw.githubusercontent.com/mc-b/lerncloud/main/services/cert-manager.sh | bash -

Als nächstes der OpenTelemetry Operator.

Dieser ermöglicht es Telemetry Daten zu erheben ohne den Applikationscode ändern zu müssen.

In [ ]:
%%bash
helm repo add open-telemetry https://open-telemetry.github.io/opentelemetry-helm-charts
helm upgrade --install opentelemetry-operator open-telemetry/opentelemetry-operator \
  -n opentelemetry \
  --create-namespace

Kontrolle ob alles läuft und CRD `instrumentation` vorhanden ist.

In [ ]:
%%bash
kubectl get pods -n opentelemetry
kubectl get crd | grep instrumentation

Services Accounts

In [ ]:
%%bash
kubectl apply -f - <<'EOF'
apiVersion: v1
kind: ServiceAccount
metadata:
  name: otel-collector
  namespace: opentelemetry
---
apiVersion: rbac.authorization.k8s.io/v1
kind: ClusterRole
metadata:
  name: otel-collector-k8sattributes
rules:
  - apiGroups: [""]
    resources: ["pods", "namespaces", "nodes"]
    verbs: ["get", "list", "watch"]
  - apiGroups: ["apps"]
    resources: ["replicasets", "deployments", "daemonsets", "statefulsets"]
    verbs: ["get", "list", "watch"]
---
apiVersion: rbac.authorization.k8s.io/v1
kind: ClusterRoleBinding
metadata:
  name: otel-collector-k8sattributes
roleRef:
  apiGroup: rbac.authorization.k8s.io
  kind: ClusterRole
  name: otel-collector-k8sattributes
subjects:
  - kind: ServiceAccount
    name: otel-collector
    namespace: opentelemetry
EOF

Prometheus, Grafana, Jaeger

In [ ]:
%%bash
helm upgrade --install my-otel-demo open-telemetry/opentelemetry-demo \
  -n opentelemetry \
  -f - <<'EOF'
components:
  accounting: { enabled: false }
  ad: { enabled: false }
  cart: { enabled: false }
  checkout: { enabled: false }
  currency: { enabled: false }
  email: { enabled: false }
  fraud-detection: { enabled: false }
  frontend: { enabled: false }
  image-provider: { enabled: false }
  load-generator: { enabled: false }
  payment: { enabled: false }
  product-catalog: { enabled: false }
  product-reviews: { enabled: false }
  quote: { enabled: false }
  recommendation: { enabled: false }
  shipping: { enabled: false }
  kafka: { enabled: false }
  llm: { enabled: false }
  postgresql: { enabled: false }
  valkey-cart: { enabled: false }

  flagd:
    enabled: true

  frontend-proxy:
    enabled: true
    envOverrides:
      - name: OTEL_COLLECTOR_NAME
        value: otel-collector-collector

default:
  envOverrides:
    - name: OTEL_COLLECTOR_NAME
      value: otel-collector-collector

opentelemetry-collector:
  enabled: false

opensearch:
  enabled: false

jaeger:
  enabled: true
  fullnameOverride: jaeger

prometheus:
  enabled: true
  server:
    fullnameOverride: prometheus
    extraFlags:
      - web.enable-otlp-receiver
      - enable-feature=exemplar-storage
    extraScrapeConfigs: |
      - job_name: kubernetes-pods-ms-otel
        kubernetes_sd_configs:
          - role: pod
            namespaces:
              names:
                - ms-otel
        relabel_configs:
          - source_labels: [__meta_kubernetes_pod_annotation_prometheus_io_scrape]
            action: keep
            regex: "true"
          - source_labels: [__meta_kubernetes_pod_annotation_prometheus_io_path]
            action: replace
            target_label: __metrics_path__
            regex: "(.+)"
          - source_labels: [__address__, __meta_kubernetes_pod_annotation_prometheus_io_port]
            action: replace
            target_label: __address__
            regex: "([^:]+)(?::\\d+)?;(\\d+)"
            replacement: "$1:$2"
          - source_labels: [__meta_kubernetes_namespace]
            action: replace
            target_label: namespace
          - source_labels: [__meta_kubernetes_pod_name]
            action: replace
            target_label: pod

      - job_name: otel-collector
        static_configs:
          - targets:
              - otel-collector-collector.opentelemetry.svc.cluster.local:8888

grafana:
  enabled: true
  fullnameOverride: grafana
  adminPassword: admin
  grafana.ini:
    auth:
      disable_login_form: true
    auth.anonymous:
      enabled: true
      org_role: Admin
    server:
      root_url: "%(protocol)s://%(domain)s:%(http_port)s/grafana"
      serve_from_sub_path: true
  sidecar:
    dashboards:
      enabled: true
    datasources:
      enabled: true
EOF

In [ ]:
%%bash
kubectl -n opentelemetry patch service frontend-proxy -p '{"spec": {"type": "LoadBalancer"}}'
kubectl -n opentelemetry patch service prometheus   -p '{"spec": {"type": "LoadBalancer"}}'
echo ""
echo "Jaeger UI         : http://"$(cat ~/data/server-ip)":"$(kubectl get service --namespace opentelemetry frontend-proxy -o=jsonpath='{ .spec.ports[0].nodePort }')/jaeger/ui
echo "Grafana           : http://"$(cat ~/data/server-ip)":"$(kubectl get service --namespace opentelemetry frontend-proxy -o=jsonpath='{ .spec.ports[0].nodePort }')/grafana
echo "Feature Flags UI  : http://"$(cat ~/data/server-ip)":"$(kubectl get service --namespace opentelemetry frontend-proxy -o=jsonpath='{ .spec.ports[0].nodePort }')/feature
echo "Prometheus UI     : http://"$(cat ~/data/server-ip):$(kubectl -n opentelemetry get service/prometheus -o=jsonpath='{ .spec.ports[0].nodePort }')

OpenCollector einrichten

In [ ]:
%%bash
kubectl apply -f - <<'EOF'
apiVersion: opentelemetry.io/v1beta1
kind: OpenTelemetryCollector
metadata:
  name: otel-collector
  namespace: opentelemetry
spec:
  mode: deployment
  serviceAccount: otel-collector
  image: otel/opentelemetry-collector-contrib:latest
  ports:
    - name: otlp-grpc
      port: 4317
      targetPort: 4317
      protocol: TCP
    - name: otlp-http
      port: 4318
      targetPort: 4318
      protocol: TCP
    - name: metrics
      port: 8888
      targetPort: 8888
      protocol: TCP
  config:
    receivers:
      otlp:
        protocols:
          grpc:
            endpoint: 0.0.0.0:4317
          http:
            endpoint: 0.0.0.0:4318

    processors:
      memory_limiter:
        check_interval: 5s
        limit_percentage: 80
        spike_limit_percentage: 25
      k8sattributes:
        auth_type: serviceAccount
        extract:
          metadata:
            - k8s.namespace.name
            - k8s.pod.name
            - k8s.deployment.name
            - k8s.node.name
      batch: {}

    exporters:
      otlp/jaeger:
        endpoint: jaeger.opentelemetry.svc.cluster.local:4317
        tls:
          insecure: true

      otlphttp/prometheus:
        endpoint: http://prometheus.opentelemetry.svc.cluster.local:9090/api/v1/otlp
        tls:
          insecure: true

      debug:
        verbosity: basic

    service:
      telemetry:
        metrics:
          readers:
            - pull:
                exporter:
                  prometheus:
                    host: 0.0.0.0
                    port: 8888
      pipelines:
        traces:
          receivers: [otlp]
          processors: [memory_limiter, k8sattributes, batch]
          exporters: [otlp/jaeger, debug]
        metrics:
          receivers: [otlp]
          processors: [memory_limiter, k8sattributes, batch]
          exporters: [otlphttp/prometheus, debug]
EOF

Kontrolle ob ein OpenTelemetryCollector Pod erstellt wurde

In [ ]:
%%bash
kubectl get opentelemetrycollector -n opentelemetry
kubectl get pods -n opentelemetry
kubectl get svc -n opentelemetry

kubectl auth can-i list pods --as=system:serviceaccount:opentelemetry:otel-collector
kubectl auth can-i list namespaces --as=system:serviceaccount:opentelemetry:otel-collector